# Cryptocurrency Pair Discovery

Goal: Find cointegrated cryptocurrency pairs on Bitvavo for statistical arbitrage.

## Strategy
We'll test pairs that are theoretically likely to be cointegrated:
1. **Major coins** (BTC, ETH, BNB, SOL, etc.) - tend to move together in bull/bear markets
2. **Layer-1 platforms** (ETH, SOL, ADA, AVAX, DOT) - competing platforms
3. **DeFi tokens** (UNI, AAVE, LINK, etc.) - sector correlation
4. **Memecoins** (DOGE, SHIB, PEPE) - speculative correlated assets

## Test Pairs
We'll fetch 90 days of hourly data and run Engle-Granger cointegration tests.

In [ ]:
import sys
sys.path.append('..')

import polars as pl
from datetime import datetime, timedelta
from statistical_arbitrage.data.bitvavo_client import BitvavoDataCollector
from statistical_arbitrage.analysis.cointegration import PairAnalysis
from statistical_arbitrage.visualization.spread_plots import plot_price_comparison
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Enable interactive tables
from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)

In [ ]:
# Initialize data collector
collector = BitvavoDataCollector()

## Define Test Pairs

We'll test multiple categories of pairs:

In [ ]:
# Define pairs to test by category
test_pairs = {
    "Major Coins": [
        ("BTC/EUR", "ETH/EUR"),  # Two dominant cryptocurrencies
        ("ETH/EUR", "BNB/EUR"),  # ETH vs Binance Smart Chain
        ("BTC/EUR", "LTC/EUR"),  # Bitcoin vs Litecoin (similar tech)
        ("BTC/EUR", "BCH/EUR"),  # Bitcoin vs Bitcoin Cash (fork)
        ("ETH/EUR", "ETC/EUR"),  # Ethereum vs Ethereum Classic (fork)
        ("BTC/EUR", "XRP/EUR"),  # Bitcoin vs Ripple
        ("ETH/EUR", "ADA/EUR"),  # ETH vs Cardano
    ],
    "Layer-1 Platforms": [
        ("ETH/EUR", "SOL/EUR"),  # Ethereum vs Solana
        ("SOL/EUR", "AVAX/EUR"), # Solana vs Avalanche
        ("ADA/EUR", "DOT/EUR"),  # Cardano vs Polkadot
        ("ATOM/EUR", "DOT/EUR"), # Cosmos vs Polkadot
        ("SOL/EUR", "NEAR/EUR"), # Solana vs Near
        ("AVAX/EUR", "NEAR/EUR"),# Avalanche vs Near
        ("ADA/EUR", "ALGO/EUR"), # Cardano vs Algorand
        ("SOL/EUR", "SUI/EUR"),  # Solana vs Sui (similar tech)
        ("AVAX/EUR", "S/EUR"),   # Avalanche vs Sonic (formerly Fantom)
    ],
    "DeFi Tokens": [
        ("UNI/EUR", "AAVE/EUR"), # Uniswap vs Aave
        ("LINK/EUR", "UNI/EUR"), # Chainlink vs Uniswap
        ("CRV/EUR", "AAVE/EUR"), # Curve vs Aave
        ("SUSHI/EUR", "UNI/EUR"),# Sushiswap vs Uniswap (competing DEXes)
        ("CRV/EUR", "SNX/EUR"),  # Curve vs Synthetix (DeFi protocols)
        ("COMP/EUR", "AAVE/EUR"),# Compound vs Aave (lending protocols)
        ("LINK/EUR", "GRT/EUR"), # Chainlink vs The Graph (oracle/data)
        ("SKY/EUR", "AAVE/EUR"), # Sky (formerly MakerDAO) vs Aave (DeFi blue chips)
    ],
    "Memecoins": [
        ("DOGE/EUR", "SHIB/EUR"), # Dogecoin vs Shiba Inu
        ("PEPE/EUR", "BONK/EUR"), # Pepe vs Bonk
        ("PEPE/EUR", "FLOKI/EUR"),# Pepe vs Floki
        ("SHIB/EUR", "BONK/EUR"), # Shiba vs Bonk
        ("DOGE/EUR", "PEPE/EUR"), # Doge vs Pepe
        ("WIF/EUR", "BONK/EUR"),  # Dogwifhat vs Bonk (Solana memes)
    ],
    "Exchange Tokens": [
        ("BNB/EUR", "FTT/EUR"),  # Binance vs FTX token
        ("BNB/EUR", "CRO/EUR"),  # Binance vs Crypto.com
        ("FTT/EUR", "CRO/EUR"),  # FTX vs Crypto.com
    ],
    "Gaming/Metaverse": [
        ("MANA/EUR", "SAND/EUR"),# Decentraland vs Sandbox
        ("AXS/EUR", "SAND/EUR"), # Axie vs Sandbox
        ("GALA/EUR", "SAND/EUR"),# Gala vs Sandbox
        ("IMX/EUR", "SAND/EUR"), # Immutable X vs Sandbox
    ],
    "Scaling/Layer-2": [
        ("POL/EUR", "OP/EUR"),   # Polygon (formerly MATIC) vs Optimism (ETH L2s)
        ("POL/EUR", "ARB/EUR"),  # Polygon vs Arbitrum (ETH L2s)
        ("OP/EUR", "ARB/EUR"),   # Optimism vs Arbitrum (competing L2s)
    ],
}

print("Testing pairs across categories:")
total_pairs = 0
for category, pairs in test_pairs.items():
    print(f"\n{category}: {len(pairs)} pairs")
    total_pairs += len(pairs)
    for p1, p2 in pairs:
        print(f"  - {p1} vs {p2}")
        
print(f"\n{'='*60}")
print(f"TOTAL: {total_pairs} pairs to test")
print(f"{'='*60}")

## Fetch Data for All Pairs

Let's fetch 90 days of hourly data for all unique assets:

In [ ]:
# Get unique assets we need to fetch
all_assets = set()
for pairs in test_pairs.values():
    for p1, p2 in pairs:
        all_assets.add(p1)
        all_assets.add(p2)

print(f"Need to fetch data for {len(all_assets)} unique assets")
print(sorted(all_assets))

In [ ]:
# Fetch data for all assets
asset_data = {}
failed_assets = []

for asset in sorted(all_assets):
    try:
        print(f"Fetching {asset}...", end=" ")
        df = collector.get_candles_range(
            market=asset,
            interval="1h",
            days_back=90
        )
        asset_data[asset] = df
        print(f"✓ ({len(df)} candles)")
    except Exception as e:
        print(f"✗ Failed: {e}")
        failed_assets.append(asset)

print(f"\nSuccessfully fetched: {len(asset_data)}/{len(all_assets)}")
if failed_assets:
    print(f"Failed assets: {failed_assets}")

## Run Cointegration Tests

Now let's test each pair for cointegration:

In [ ]:
# Function to test a pair and return results
def test_pair_cointegration(asset1: str, asset2: str, data1: pl.DataFrame, data2: pl.DataFrame) -> dict:
    """Test if two assets are cointegrated."""
    # Align data by timestamp (in case of missing candles)
    merged = data1.join(data2, on="datetime", how="inner", suffix="_2")
    
    # Extract close prices
    prices1 = merged["close"]
    prices2 = merged["close_2"]
    
    # Run analysis
    analysis = PairAnalysis(prices1, prices2)
    coint_results = analysis.test_cointegration()
    
    # Calculate correlation for comparison
    import numpy as np
    correlation = np.corrcoef(prices1.to_numpy(), prices2.to_numpy())[0, 1]
    
    return {
        "asset1": asset1,
        "asset2": asset2,
        "data_points": len(merged),
        "correlation": correlation,
        "p_value": coint_results["p_value"],
        "is_cointegrated": coint_results["is_cointegrated"],
        "hedge_ratio": coint_results["hedge_ratio"],
        "adf_statistic": coint_results["spread_stationarity"]["adf_statistic"],
        "half_life": analysis.calculate_half_life() if coint_results["is_cointegrated"] else None,
    }

# Test all pairs
results = []

for category, pairs in test_pairs.items():
    print(f"\n{'='*60}")
    print(f"Testing {category}")
    print(f"{'='*60}")
    
    for asset1, asset2 in pairs:
        # Skip if data not available
        if asset1 not in asset_data or asset2 not in asset_data:
            print(f"\n{asset1} vs {asset2}: SKIPPED (data not available)")
            continue
        
        try:
            result = test_pair_cointegration(asset1, asset2, asset_data[asset1], asset_data[asset2])
            result["category"] = category
            results.append(result)
            
            # Print result
            status = "✅ COINTEGRATED" if result["is_cointegrated"] else "❌ Not cointegrated"
            print(f"\n{asset1} vs {asset2}: {status}")
            print(f"  Correlation: {result['correlation']:.4f}")
            print(f"  P-value: {result['p_value']:.6f}")
            print(f"  Hedge ratio: {result['hedge_ratio']:.6f}")
            if result['half_life']:
                print(f"  Half-life: {result['half_life']:.2f} hours")
        except Exception as e:
            print(f"\n{asset1} vs {asset2}: ERROR - {e}")

In [ ]:
# Convert results to DataFrame
results_df = pl.DataFrame(results)

# Show all results sorted by p-value (lower = more cointegrated)
print("\n" + "="*80)
print("ALL RESULTS (sorted by p-value)")
print("="*80)

display_df = results_df.select([
    "category",
    "asset1",
    "asset2",
    "is_cointegrated",
    "p_value",
    "correlation",
    "hedge_ratio",
    "half_life",
]).sort("p_value")

# Interactive table - just display the dataframe
display_df

In [ ]:
# Count cointegrated pairs
cointegrated_count = results_df.filter(pl.col("is_cointegrated").cast(pl.Boolean)).shape[0]
total_count = results_df.shape[0]

print(f"✅ Found {cointegrated_count} cointegrated pairs out of {total_count} tested")

In [ ]:
display(display_df)

# Get cointegrated pairs sorted by p-value
cointegrated_pairs = results_df.filter(pl.col("is_cointegrated").cast(pl.Boolean)).sort("p_value")

if len(cointegrated_pairs) > 0:
    print(f"Plotting {len(cointegrated_pairs)} cointegrated pair(s):\n")
    
    for row in cointegrated_pairs.iter_rows(named=True):
        asset1 = row["asset1"]
        asset2 = row["asset2"]
        
        # Get data and align
        merged = asset_data[asset1].join(asset_data[asset2], on="datetime", how="inner", suffix="_2")
        
        # Create plot
        fig = plot_price_comparison(
            dates=merged["datetime"].to_list(),
            asset1_prices=merged["close"].to_list(),
            asset2_prices=merged["close_2"].to_list(),
            asset1_name=asset1.replace("/EUR", ""),
            asset2_name=asset2.replace("/EUR", ""),
        )
        
        # Add p-value to title
        fig.update_layout(
            title=f"{asset1} vs {asset2} - P-value: {row['p_value']:.6f}, Half-life: {row['half_life']:.1f}h"
        )
        
        fig.show()
else:
    print("No cointegrated pairs found. Try testing more pairs or different timeframes.")

## Next Steps

Based on the results:

1. **If we found cointegrated pairs**: 
   - Visualize the spread and z-score
   - Design trading signals (e.g., z-score > 2 = short spread)
   - Build backtesting framework

2. **If no pairs found**:
   - Test different timeframes (5min, 15min, 4h, daily)
   - Test more exotic pairs
   - Consider other exchanges with more trading pairs

3. **Analysis**:
   - Compare correlation vs cointegration
   - Understand why certain pairs are/aren't cointegrated
   - Check if cointegration is stable over time (rolling window tests)